In [ ]:
"""Manufactured SM1.ipynb

"""

In [ ]:
import argparse
import copy
import csv
import json
import math
import random
import sys
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Literal, NamedTuple, Tuple

In [ ]:
import matplotlib

In [ ]:
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn

In [ ]:
Tensor = torch.Tensor
Method = Literal["PINN", "CRVPINN"]
FINAL_TIME = 1.0
alfa = 1.0
epsilon = 0.1

In [ ]:
@dataclass
class Config:
    epochs: int = 20000
    n_space: int = 50
    n_time: int = 25
    n_eval_space: int = 25
    n_eval_time: int = 25
    width: int = 64
    hidden_layers: int = 3
    learning_rate: float = 1.0e-3
    lr_decay: float = 0.9995
    log_every: int = 50
    plot_points: int = 121
    seed: int = 1234
    dtype: str = "float64"
    device: str = "auto"
    output_dir: str = "advection_diffusion_corrected_results"
    cpu_threads: int = 4
    normalize_objective: bool = True

In [ ]:
class Grid(NamedTuple):
    x: Tensor
    y: Tensor
    t: Tensor
    h: float
    dt: float
    n_space: int
    n_time: int

In [ ]:
def parse_args() -> Config:
    parser = argparse.ArgumentParser(
        description="Poprawione porównanie PINN i CRVPINN.",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    parser.add_argument("--epochs", type=int, default=Config.epochs)
    parser.add_argument("--n-space", type=int, default=Config.n_space)
    parser.add_argument("--n-time", type=int, default=Config.n_time)
    parser.add_argument("--n-eval-space", type=int, default=Config.n_eval_space)
    parser.add_argument("--n-eval-time", type=int, default=Config.n_eval_time)
    parser.add_argument("--width", type=int, default=Config.width)
    parser.add_argument("--hidden-layers", type=int, default=Config.hidden_layers)
    parser.add_argument("--learning-rate", type=float, default=Config.learning_rate)
    parser.add_argument("--lr-decay", type=float, default=Config.lr_decay)
    parser.add_argument("--log-every", type=int, default=Config.log_every)
    parser.add_argument("--plot-points", type=int, default=Config.plot_points)
    parser.add_argument("--seed", type=int, default=Config.seed)
    parser.add_argument(
        "--dtype", choices=("float32", "float64"), default=Config.dtype
    )
    parser.add_argument(
        "--device", choices=("auto", "cpu", "cuda"), default=Config.device
    )
    parser.add_argument("--output-dir", type=str, default=Config.output_dir)
    parser.add_argument("--cpu-threads", type=int, default=Config.cpu_threads)
    parser.add_argument(
        "--no-normalize-objective",
        action="store_false",
        dest="normalize_objective",
        help="Nie normalizuj funkcji celu przez jej wartość początkową.",
    )
    parser.set_defaults(normalize_objective=Config.normalize_objective)

    # In a Colab/Jupyter environment, a -f argument is often passed by the kernel.
    # This causes argparse to fail as it's not a defined argument in this script.
    # We check for this and pass an empty list if detected, effectively ignoring kernel args.
    args = sys.argv[1:]
    if '-f' in args:
        # Remove -f and its value from args, or just parse an empty list
        args = []

    return Config(**vars(parser.parse_args(args=args)))

In [ ]:
def select_device(name: str) -> torch.device:
    if name == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if name == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("Wybrano CUDA, ale CUDA nie jest dostępna.")
    return torch.device(name)

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [ ]:
def phi(x: Tensor) -> Tensor :
  one_over_epsilon_tensor = torch.tensor(1.0 / epsilon, device=x.device, dtype=x.dtype)
  return (
      -1 * ( x * (1 - torch.exp(one_over_epsilon_tensor) ) + torch.exp( x/epsilon ) -1 ) / ( torch.exp(one_over_epsilon_tensor) - 1)
  )

In [ ]:
def phi_dx(x: Tensor) -> Tensor :
  one_over_epsilon_tensor = torch.tensor(1.0 / epsilon, device=x.device, dtype=x.dtype)
  return (
      1 -  one_over_epsilon_tensor * ( torch.exp( x/epsilon )  ) / ( torch.exp(one_over_epsilon_tensor) - 1 )
  )

In [ ]:
def psi( x: Tensor ) -> Tensor :
  one_over_epsilon_tensor = torch.tensor(1.0 / epsilon, device=x.device, dtype=x.dtype)
  return (
      -1 * one_over_epsilon_tensor * ( torch.exp( x/epsilon ) -1 ) / ( torch.exp(one_over_epsilon_tensor) - 1 )
  )

In [ ]:
def u_exact(x: Tensor, y: Tensor, t: Tensor) -> Tensor:
    return (
        phi(x)
        * torch.sin(math.pi * y)
        * torch.exp(-alfa* t)
    )

In [ ]:
def initial_condition(x: Tensor, y: Tensor) -> Tensor:
    return  phi(x) * torch.sin(math.pi * y) # torch.sin(math.pi * x) *

In [ ]:
def forcing(x: Tensor, y: Tensor, t: Tensor) -> Tensor:
    return (
        torch.exp(-t * alfa) * ( torch.sin(math.pi * y) * ( 1 + ( epsilon * math.pi * math.pi - alfa) * phi(x) ) + math.pi * torch.cos(math.pi * y) * phi(x) )
    )

    #sx = torch.sin(math.pi * x)
    #sy = torch.sin(math.pi * y)
    #cx = torch.cos(math.pi * xh
    #cy = torch.cos(math.pi * y)
    #
    #return torch.exp(-t * alfa) * (
    #    (2.0 * epsilon *  math.pi**2 - 1.0 * alfa) * sx * sy
    #    + math.pi * cx * sy
    #    + math.pi * sx * cy
    #)

In [ ]:
def exact_derivatives(
    x: Tensor, y: Tensor, t: Tensor
) -> Tuple[Tensor, Tensor, Tensor]:
    exp_t = torch.exp(-t * alfa )
    ux = (
        phi_dx(x)
        #math.pi
        #* torch.cos(math.pi * x)
        * torch.sin(math.pi * y)
        * exp_t
    )
    uy = (
        math.pi
        * phi(x)
        #* torch.sin(math.pi * x)
        * torch.cos(math.pi * y)
        * exp_t
    )
    ut = -u_exact(x, y, t) * alfa
    return ux, uy, ut

In [ ]:
class MLP(nn.Module):
    def __init__(self, hidden_layers: int, width: int) -> None:
        super().__init__()

        layers: List[nn.Module] = [nn.Linear(3, width), nn.Tanh()]
        for _ in range(hidden_layers - 1):
            layers.extend([nn.Linear(width, width), nn.Tanh()])
        layers.append(nn.Linear(width, 1))

        self.net = nn.Sequential(*layers)
        self.reset_parameters()

    def reset_parameters(self) -> None:
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_normal_(module.weight)
                nn.init.zeros_(module.bias)

    def forward(self, x: Tensor, y: Tensor, t: Tensor) -> Tensor:
        raw = self.net(torch.cat((x, y, t), dim=1))

        # Dokładne narzucenie warunków:
        # t=0 -> u=u0,
        # x=0,1 lub y=0,1 -> u=0.
        boundary_factor = x * (1.0 - x) * y * (1.0 - y)
        return initial_condition(x, y) + t * boundary_factor * raw

In [ ]:
def make_grid(
    n_space: int,
    n_time: int,
    device: torch.device,
    dtype: torch.dtype,
    requires_grad: bool = True,
) -> Grid:
    if n_space < 2 or n_time < 2:
        raise ValueError("n_space i n_time muszą być co najmniej równe 2.")

    # W przestrzeni: punkty wewnętrzne dla warunków Dirichleta.
    h = 1.0 / (n_space + 1)
    spatial_axis = (
        torch.arange(1, n_space + 1, device=device, dtype=dtype) * h
    )

    # W czasie: punkty środkowe przedziałów.
    dt = FINAL_TIME / n_time
    time_axis = (
        torch.arange(n_time, device=device, dtype=dtype) + 0.5
    ) * dt

    x, y, t = torch.meshgrid(
        spatial_axis, spatial_axis, time_axis, indexing="ij"
    )

    x = x.reshape(-1, 1).clone().detach().requires_grad_(requires_grad)
    y = y.reshape(-1, 1).clone().detach().requires_grad_(requires_grad)
    t = t.reshape(-1, 1).clone().detach().requires_grad_(requires_grad)

    return Grid(
        x=x,
        y=y,
        t=t,
        h=float(h),
        dt=float(dt),
        n_space=n_space,
        n_time=n_time,
    )

In [ ]:
def pde_residual(
    model: nn.Module, x: Tensor, y: Tensor, t: Tensor
) -> Tensor:
    u = model(x, y, t)
    ones = torch.ones_like(u)

    ux, uy, ut = torch.autograd.grad(
        u,
        (x, y, t),
        grad_outputs=ones,
        create_graph=True,
        retain_graph=True,
    )

    uxx = torch.autograd.grad(
        ux,
        x,
        grad_outputs=torch.ones_like(ux),
        create_graph=True,
        retain_graph=True,
    )[0]

    uyy = torch.autograd.grad(
        uy,
        y,
        grad_outputs=torch.ones_like(uy),
        create_graph=True,
        retain_graph=True,
    )[0]

    return ut + ux + uy - epsilon * uxx - epsilon * uyy - forcing(x, y, t)

In [ ]:
class SpatialHMinusOne2D(nn.Module):
    """
    Dyskretna przestrzenna norma H^{-1}(Omega).

    Dla każdego czasu rozwiązuje:
        -Delta_h z = rhs
    na wewnętrznej siatce z zerowymi warunkami Dirichleta.

    Operator -Delta_h zawiera skalę 1/h^2.
    """

    def __init__(
        self,
        n_space: int,
        h: float,
        device: torch.device,
        dtype: torch.dtype,
    ) -> None:
        super().__init__()

        idx = torch.arange(
            1, n_space + 1, device=device, dtype=dtype
        )

        # Ortonormalna dyskretna transformata sinusowa.
        q = torch.sqrt(
            torch.tensor(
                2.0 / (n_space + 1),
                device=device,
                dtype=dtype,
            )
        ) * torch.sin(
            math.pi
            * idx[:, None]
            * idx[None, :]
            / (n_space + 1)
        )

        lam_1d = (
            2.0
            - 2.0 * torch.cos(math.pi * idx / (n_space + 1))
        ) / (h**2)

        eig_2d = lam_1d[:, None] + lam_1d[None, :]

        self.n_space = n_space
        self.h = float(h)
        self.register_buffer("q", q)
        self.register_buffer("eig_2d", eig_2d)

    def solve_poisson(self, rhs: Tensor) -> Tensor:
        """
        rhs: [n_time, n_space, n_space]
        wynik ma ten sam kształt.
        """
        if rhs.ndim != 3:
            raise ValueError(
                "rhs musi mieć kształt [n_time, n_space, n_space]."
            )

        spectral = torch.matmul(self.q.T, rhs)
        spectral = torch.matmul(spectral, self.q)
        spectral = spectral / self.eig_2d

        solution = torch.matmul(self.q, spectral)
        solution = torch.matmul(solution, self.q.T)
        return solution

    def per_time_dual_norm_sq(self, rhs: Tensor) -> Tensor:
        """
        Zwraca ||rhs(t_k)||^2_{H^{-1}} dla każdego czasu t_k.
        """
        riesz_lift = self.solve_poisson(rhs)
        values = self.h**2 * torch.sum(
            rhs * riesz_lift, dim=(1, 2)
        )
        return torch.clamp(values, min=0.0)

    def space_time_dual_norm_sq(self, rhs: Tensor, dt: float) -> Tensor:
        """
        Przybliża integral_0^T ||rhs(t)||^2_{H^{-1}} dt.
        """
        return dt * torch.sum(self.per_time_dual_norm_sq(rhs))

In [ ]:
def residual_as_time_batch(residual: Tensor, grid: Grid) -> Tensor:
    """
    Kolejność meshgrid to [x, y, t].
    Zmieniamy na [t, x, y].
    """
    return residual.reshape(
        grid.n_space, grid.n_space, grid.n_time
    ).permute(2, 0, 1)

In [ ]:
def compute_raw_losses(
    model: nn.Module,
    grid: Grid,
    spatial_dual: SpatialHMinusOne2D,
) -> Tuple[Tensor, Tensor]:
    residual = pde_residual(model, grid.x, grid.y, grid.t)
    residual_txy = residual_as_time_batch(residual, grid)

    # Fizyczna, dyskretna norma L2 w przestrzeni i czasie.
    pinn_loss = (
        grid.dt
        * grid.h**2
        * torch.sum(residual_txy.square())
    )

    # Poprawiona norma dualna: H^{-1} tylko w przestrzeni,
    # następnie całkowanie po czasie.
    crvpinn_loss = spatial_dual.space_time_dual_norm_sq(
        residual_txy, grid.dt
    )

    return pinn_loss, crvpinn_loss

In [ ]:
def true_errors(
    model: nn.Module,
    grid: Grid,
    spatial_dual: SpatialHMinusOne2D,
) -> Dict[str, float]:
    u = model(grid.x, grid.y, grid.t)

    ux, uy, ut = torch.autograd.grad(
        u,
        (grid.x, grid.y, grid.t),
        grad_outputs=torch.ones_like(u),
        create_graph=False,
        retain_graph=True,
    )

    ue = u_exact(grid.x, grid.y, grid.t)
    uex, uey, uet = exact_derivatives(grid.x, grid.y, grid.t)

    ex = ux - uex
    ey = uy - uey
    et = ut - uet

    ex_txy = residual_as_time_batch(ex, grid)
    ey_txy = residual_as_time_batch(ey, grid)
    et_txy = residual_as_time_batch(et, grid)

    spatial_gradient_error_sq_per_time = (
        grid.h**2
        * torch.sum(
            ex_txy.square() + ey_txy.square(),
            dim=(1, 2),
        )
    )

    time_derivative_error_sq_per_time = (
        spatial_dual.per_time_dual_norm_sq(et_txy)
    )

    spatial_gradient_error_sq = (
        grid.dt * torch.sum(spatial_gradient_error_sq_per_time)
    )
    time_derivative_error_sq = (
        grid.dt * torch.sum(time_derivative_error_sq_per_time)
    )

    parabolic_error_sq = (
        spatial_gradient_error_sq + time_derivative_error_sq
    )

    error_l2_sq = (
        grid.dt
        * grid.h**2
        * torch.sum((u - ue).square())
    )
    exact_l2_sq = (
        grid.dt
        * grid.h**2
        * torch.sum(ue.square())
    )

    return {
        "true_parabolic_error": float(
            torch.sqrt(torch.clamp(parabolic_error_sq, min=0.0))
            .detach()
            .cpu()
        ),
        "true_spatial_gradient_error": float(
            torch.sqrt(
                torch.clamp(spatial_gradient_error_sq, min=0.0)
            )
            .detach()
            .cpu()
        ),
        "true_time_hminus1_error": float(
            torch.sqrt(
                torch.clamp(time_derivative_error_sq, min=0.0)
            )
            .detach()
            .cpu()
        ),
        "relative_l2_error": float(
            torch.sqrt(
                torch.clamp(
                    error_l2_sq / exact_l2_sq,
                    min=0.0,
                )
            )
            .detach()
            .cpu()
        ),
    }

In [ ]:
def evaluate_metrics(
    model: nn.Module,
    train_grid: Grid,
    train_dual: SpatialHMinusOne2D,
    eval_grid: Grid,
    eval_dual: SpatialHMinusOne2D,
) -> Dict[str, float]:
    model.eval()

    # Residuum nadal wymaga autograd.
    pinn_loss, crv_loss = compute_raw_losses(
        model, train_grid, train_dual
    )
    errors = true_errors(model, eval_grid, eval_dual)

    result = {
        "pinn_loss": float(pinn_loss.detach().cpu()),
        "crvpinn_loss": float(crv_loss.detach().cpu()),
        "pinn_residual_l2": float(
            torch.sqrt(torch.clamp(pinn_loss, min=0.0))
            .detach()
            .cpu()
        ),
        "crvpinn_estimator": float(
            torch.sqrt(torch.clamp(crv_loss, min=0.0))
            .detach()
            .cpu()
        ),
    }
    result.update(errors)
    return result

In [ ]:
def train(
    model: nn.Module,
    method: Method,
    train_grid: Grid,
    train_dual: SpatialHMinusOne2D,
    eval_grid: Grid,
    eval_dual: SpatialHMinusOne2D,
    cfg: Config,
) -> List[Dict[str, float]]:
    optimizer = torch.optim.Adam(
        model.parameters(), lr=cfg.learning_rate
    )
    scheduler = torch.optim.lr_scheduler.ExponentialLR(
        optimizer, gamma=cfg.lr_decay
    )

    # Normalizacja nie zmienia minimum funkcji celu.
    # Ujednolica natomiast początkową skalę gradientów obu metod.
    pinn_0, crv_0 = compute_raw_losses(
        model, train_grid, train_dual
    )
    initial_raw_objective = (
        pinn_0 if method == "PINN" else crv_0
    ).detach()

    if cfg.normalize_objective:
        objective_scale = torch.clamp(
            initial_raw_objective,
            min=torch.finfo(initial_raw_objective.dtype).eps,
        )
    else:
        objective_scale = torch.ones_like(initial_raw_objective)

    history: List[Dict[str, float]] = []
    start = time.perf_counter()

    def log(epoch: int) -> None:
        metrics = evaluate_metrics(
            model,
            train_grid,
            train_dual,
            eval_grid,
            eval_dual,
        )

        raw_objective = (
            metrics["pinn_loss"]
            if method == "PINN"
            else metrics["crvpinn_loss"]
        )

        metrics.update(
            {
                "model": method,
                "epoch": epoch,
                "seconds": time.perf_counter() - start,
                "learning_rate": optimizer.param_groups[0]["lr"],
                "raw_train_objective": raw_objective,
                "normalized_train_objective": (
                    raw_objective
                    / float(objective_scale.detach().cpu())
                ),
            }
        )
        history.append(metrics)

        print(
            f"[{method:8s}] "
            f"epoch={epoch:6d}  "
            f"sqrt(loss)={math.sqrt(max(raw_objective, 0.0)):.4e}  "
            f"true W-error={metrics['true_parabolic_error']:.4e}  "
            f"rel L2={metrics['relative_l2_error']:.4e}"
        )

    log(0)
    model.train()

    for epoch in range(1, cfg.epochs + 1):
        optimizer.zero_grad(set_to_none=True)

        pinn_loss, crv_loss = compute_raw_losses(
            model, train_grid, train_dual
        )
        raw_objective = (
            pinn_loss if method == "PINN" else crv_loss
        )
        objective = raw_objective / objective_scale

        if not torch.isfinite(objective):
            raise FloatingPointError(
                f"Niefinitywna funkcja celu dla {method}, "
                f"epoka {epoch}."
            )

        objective.backward()
        optimizer.step()
        scheduler.step()

        if epoch % cfg.log_every == 0 or epoch == cfg.epochs:
            log(epoch)
            model.train()

    return history

In [ ]:
def save_history(
    histories: Dict[Method, List[Dict[str, float]]],
    output_dir: Path,
) -> None:
    rows = histories["PINN"] + histories["CRVPINN"]

    fieldnames = [
        "model",
        "epoch",
        "seconds",
        "learning_rate",
        "raw_train_objective",
        "normalized_train_objective",
        "pinn_loss",
        "crvpinn_loss",
        "pinn_residual_l2",
        "crvpinn_estimator",
        "true_parabolic_error",
        "true_spatial_gradient_error",
        "true_time_hminus1_error",
        "relative_l2_error",
    ]

    with (output_dir / "history.csv").open(
        "w", newline="", encoding="utf-8"
    ) as handle:
        writer = csv.DictWriter(
            handle, fieldnames=fieldnames
        )
        writer.writeheader()
        writer.writerows(rows)

In [ ]:
def history_arrays(
    history: List[Dict[str, float]],
) -> Dict[str, np.ndarray]:
    keys = history[0].keys()
    return {
        key: np.asarray([row[key] for row in history])
        for key in keys
        if key != "model"
    }

In [ ]:
def plot_convergence(
    histories: Dict[Method, List[Dict[str, float]]],
    output_dir: Path,
) -> None:
    hp = history_arrays(histories["PINN"])
    hc = history_arrays(histories["CRVPINN"])

    fig, ax = plt.subplots(figsize=(8, 6), dpi=140)
    ax.semilogy(
        hp["epoch"],
        hp["pinn_residual_l2"],
        label=r"PINN: $||R||_{L^2_tL^2_x}$",
    )
    ax.semilogy(
        hp["epoch"],
        hp["true_parabolic_error"],
        label="PINN: true parabolic error",
    )
    ax.set_xlabel("Iteracja")
    ax.set_ylabel("Wartość")
    ax.set_title("PINN: sqrt(loss) a true error")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(output_dir / "convergence_PINN.png")
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(8, 6), dpi=140)
    ax.semilogy(
        hc["epoch"],
        hc["crvpinn_estimator"],
        label=r"CRVPINN: $||R||_{L^2_tH^{-1}_x}$",
    )
    ax.semilogy(
        hc["epoch"],
        hc["true_parabolic_error"],
        label="CRVPINN: true parabolic error",
    )
    ax.set_xlabel("Iteracja")
    ax.set_ylabel("Wartość")
    ax.set_title("CRVPINN: sqrt(robust loss) a true error")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(output_dir / "convergence_CRVPINN.png")
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(8, 6), dpi=140)
    ax.semilogy(
        hp["epoch"],
        hp["true_parabolic_error"],
        label="PINN",
    )
    ax.semilogy(
        hc["epoch"],
        hc["true_parabolic_error"],
        label="CRVPINN",
    )
    ax.set_xlabel("Iteracja")
    ax.set_ylabel("True parabolic error")
    ax.set_title("Porównanie zbieżności true error")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(output_dir / "true_error_comparison.png")
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(8, 6), dpi=140)
    ax.semilogy(
        hp["seconds"],
        hp["true_parabolic_error"],
        label="PINN",
    )
    ax.semilogy(
        hc["seconds"],
        hc["true_parabolic_error"],
        label="CRVPINN",
    )
    ax.set_xlabel("Czas obliczeń [s]")
    ax.set_ylabel("True parabolic error")
    ax.set_title("True error względem czasu obliczeń")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(output_dir / "true_error_vs_time.png")
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(8, 6), dpi=140)
    ax.semilogy(
        hp["epoch"],
        hp["relative_l2_error"],
        label="PINN",
    )
    ax.semilogy(
        hc["epoch"],
        hc["relative_l2_error"],
        label="CRVPINN",
    )
    ax.set_xlabel("Iteracja")
    ax.set_ylabel("Względny błąd L2")
    ax.set_title("Porównanie względnego błędu L2")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(output_dir / "relative_L2_comparison.png")
    plt.close(fig)

    eps = np.finfo(float).tiny
    fig, ax = plt.subplots(figsize=(8, 6), dpi=140)
    ax.semilogy(
        hp["epoch"],
        hp["pinn_residual_l2"]
        / np.maximum(hp["true_parabolic_error"], eps),
        label="PINN: sqrt(loss) / true error",
    )
    ax.semilogy(
        hc["epoch"],
        hc["crvpinn_estimator"]
        / np.maximum(hc["true_parabolic_error"], eps),
        label="CRVPINN: sqrt(loss) / true error",
    )
    ax.axhline(1.0, linewidth=1.0, linestyle="--")
    ax.set_xlabel("Iteracja")
    ax.set_ylabel("Iloraz")
    ax.set_title("Skala sqrt(loss) względem true error")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(output_dir / "loss_to_true_error_ratio.png")
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(8, 6), dpi=140)
    ax.semilogy(
        hp["epoch"],
        hp["true_spatial_gradient_error"],
        label="PINN: spatial gradient part",
    )
    ax.semilogy(
        hp["epoch"],
        hp["true_time_hminus1_error"],
        label="PINN: time H^-1 part",
    )
    ax.semilogy(
        hc["epoch"],
        hc["true_spatial_gradient_error"],
        label="CRVPINN: spatial gradient part",
    )
    ax.semilogy(
        hc["epoch"],
        hc["true_time_hminus1_error"],
        label="CRVPINN: time H^-1 part",
    )
    ax.set_xlabel("Iteracja")
    ax.set_ylabel("Składnik błędu")
    ax.set_title("Składniki parabolicznego true error")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(output_dir / "true_error_components.png")
    plt.close(fig)

In [ ]:
def evaluate_slice(
    model: nn.Module,
    t_value: float,
    points: int,
    device: torch.device,
    dtype: torch.dtype,
) -> Tuple[np.ndarray, ...]:
    axis = torch.linspace(
        0.0, 1.0, points, device=device, dtype=dtype
    )
    x, y = torch.meshgrid(axis, axis, indexing="ij")
    xf = x.reshape(-1, 1)
    yf = y.reshape(-1, 1)
    tf = torch.full_like(xf, t_value)

    with torch.no_grad():
        pred = model(xf, yf, tf).reshape(points, points)
        exact = u_exact(xf, yf, tf).reshape(points, points)

    error = torch.abs(pred - exact)

    return (
        x.detach().cpu().numpy(),
        y.detach().cpu().numpy(),
        exact.detach().cpu().numpy(),
        pred.detach().cpu().numpy(),
        error.detach().cpu().numpy(),
    )

In [ ]:
def plot_solution_slices(
    model: nn.Module,
    method: Method,
    output_dir: Path,
    points: int,
    device: torch.device,
    dtype: torch.dtype,
) -> None:
    times = (0.25, 0.50, 1.00)
    fig, axes = plt.subplots(
        len(times),
        3,
        figsize=(13, 11),
        dpi=140,
        constrained_layout=True,
    )

    for row, t_value in enumerate(times):
        x, y, exact, pred, error = evaluate_slice(
            model, t_value, points, device, dtype
        )

        fields = (exact, pred, error)
        titles = ("exact", method, "|error|")

        for col, (field, title) in enumerate(
            zip(fields, titles)
        ):
            mesh = axes[row, col].pcolormesh(
                x, y, field, shading="auto"
            )
            axes[row, col].set_aspect("equal")
            axes[row, col].set_xlabel("x")
            axes[row, col].set_ylabel("y")
            axes[row, col].set_title(
                f"{title}, t={t_value:.2f}"
            )
            fig.colorbar(
                mesh, ax=axes[row, col], shrink=0.82
            )

    fig.suptitle(f"Rozwiązanie i błąd — {method}")
    fig.savefig(
        output_dir / f"solution_slices_{method}.png"
    )
    plt.close(fig)

In [ ]:
def plot_centerline(
    models: Dict[Method, nn.Module],
    output_dir: Path,
    device: torch.device,
    dtype: torch.dtype,
) -> None:
    x = torch.linspace(
        0.0, 1.0, 401, device=device, dtype=dtype
    ).reshape(-1, 1)
    y = torch.full_like(x, 0.5)
    times = (0.25, 0.50, 1.00)

    fig, axes = plt.subplots(
        1,
        len(times),
        figsize=(15, 4.2),
        dpi=140,
        constrained_layout=True,
    )

    for ax, t_value in zip(axes, times):
        t = torch.full_like(x, t_value)

        with torch.no_grad():
            exact = u_exact(x, y, t).cpu().numpy()
            pred_pinn = models["PINN"](x, y, t).cpu().numpy()
            pred_crv = models["CRVPINN"](x, y, t).cpu().numpy()

        x_np = x.cpu().numpy()

        ax.plot(x_np, exact, label="exact", linewidth=2.0)
        ax.plot(x_np, pred_pinn, label="PINN")
        ax.plot(x_np, pred_crv, label="CRVPINN")
        ax.set_xlabel("x")
        ax.set_ylabel("u(x,0.5,t)")
        ax.set_title(f"t={t_value:.2f}")
        ax.grid(True, alpha=0.3)

    axes[-1].legend()
    fig.suptitle("Przekrój y=0.5")
    fig.savefig(output_dir / "centerline_comparison.png")
    plt.close(fig)

In [ ]:
def save_summary(
    histories: Dict[Method, List[Dict[str, float]]],
    output_dir: Path,
) -> None:
    summary = {
        method: histories[method][-1]
        for method in ("PINN", "CRVPINN")
    }

    with (output_dir / "final_metrics.json").open(
        "w", encoding="utf-8"
    ) as handle:
        json.dump(
            summary,
            handle,
            indent=2,
            ensure_ascii=False,
        )

    lines = [
        "WYNIKI KOŃCOWE",
        "==============",
        "",
    ]

    for method in ("PINN", "CRVPINN"):
        row = summary[method]
        lines.extend(
            [
                method,
                f"  epoch: {int(row['epoch'])}",
                f"  sqrt(raw loss): "
                f"{math.sqrt(max(row['raw_train_objective'], 0.0)):.8e}",
                f"  true parabolic error: "
                f"{row['true_parabolic_error']:.8e}",
                f"  relative L2 error: "
                f"{row['relative_l2_error']:.8e}",
                f"  czas: {row['seconds']:.2f} s",
                "",
            ]
        )

    (output_dir / "summary.txt").write_text(
        "\n".join(lines), encoding="utf-8"
    )

In [ ]:
def main() -> None:
    cfg = parse_args()

    if cfg.epochs < 1:
        raise ValueError("--epochs musi być dodatnie.")
    if cfg.log_every < 1:
        raise ValueError("--log-every musi być dodatnie.")

    set_seed(cfg.seed)
    device = select_device(cfg.device)

    if device.type == "cpu":
        torch.set_num_threads(max(1, cfg.cpu_threads))

    dtype = (
        torch.float32
        if cfg.dtype == "float32"
        else torch.float64
    )
    torch.set_default_dtype(dtype)

    output_dir = Path(cfg.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    with (output_dir / "config.json").open(
        "w", encoding="utf-8"
    ) as handle:
        json.dump(
            asdict(cfg),
            handle,
            indent=2,
            ensure_ascii=False,
        )

    print(
        f"Device: {device}; dtype: {dtype}; "
        f"output: {output_dir.resolve()}"
    )
    print(
        "CRVPINN używa H^{-1} tylko w przestrzeni; "
        "czas jest całkowany osobno."
    )

    train_grid = make_grid(
        cfg.n_space,
        cfg.n_time,
        device,
        dtype,
        requires_grad=True,
    )
    eval_grid = make_grid(
        cfg.n_eval_space,
        cfg.n_eval_time,
        device,
        dtype,
        requires_grad=True,
    )

    train_dual = SpatialHMinusOne2D(
        cfg.n_space,
        train_grid.h,
        device,
        dtype,
    ).to(device)

    eval_dual = SpatialHMinusOne2D(
        cfg.n_eval_space,
        eval_grid.h,
        device,
        dtype,
    ).to(device)

    # Obie metody zaczynają z dokładnie tych samych wag.
    base_model = MLP(
        cfg.hidden_layers, cfg.width
    ).to(device=device, dtype=dtype)
    initial_state = copy.deepcopy(base_model.state_dict())

    models: Dict[Method, nn.Module] = {
        "PINN": MLP(
            cfg.hidden_layers, cfg.width
        ).to(device=device, dtype=dtype),
        "CRVPINN": MLP(
            cfg.hidden_layers, cfg.width
        ).to(device=device, dtype=dtype),
    }

    models["PINN"].load_state_dict(initial_state)
    models["CRVPINN"].load_state_dict(initial_state)

    histories: Dict[
        Method, List[Dict[str, float]]
    ] = {
        "PINN": train(
            models["PINN"],
            "PINN",
            train_grid,
            train_dual,
            eval_grid,
            eval_dual,
            cfg,
        ),
        "CRVPINN": train(
            models["CRVPINN"],
            "CRVPINN",
            train_grid,
            train_dual,
            eval_grid,
            eval_dual,
            cfg,
        ),
    }

    torch.save(
        models["PINN"].state_dict(),
        output_dir / "model_PINN.pt",
    )
    torch.save(
        models["CRVPINN"].state_dict(),
        output_dir / "model_CRVPINN.pt",
    )

    save_history(histories, output_dir)
    plot_convergence(histories, output_dir)

    plot_solution_slices(
        models["PINN"],
        "PINN",
        output_dir,
        cfg.plot_points,
        device,
        dtype,
    )
    plot_solution_slices(
        models["CRVPINN"],
        "CRVPINN",
        output_dir,
        cfg.plot_points,
        device,
        dtype,
    )
    plot_centerline(
        models, output_dir, device, dtype
    )
    save_summary(histories, output_dir)

    print("\nGotowe. Najważniejsze pliki:")
    for name in (
        "history.csv",
        "summary.txt",
        "final_metrics.json",
        "convergence_PINN.png",
        "convergence_CRVPINN.png",
        "true_error_comparison.png",
        "true_error_vs_time.png",
        "relative_L2_comparison.png",
        "loss_to_true_error_ratio.png",
        "true_error_components.png",
        "solution_slices_PINN.png",
        "solution_slices_CRVPINN.png",
        "centerline_comparison.png",
    ):
        print(f"  {output_dir / name}")

In [ ]:
if __name__ == "__main__":
    main()